## Preprocessing for covariance analysis

In this notebook, we are implementing preprcessing before covariance analysis using steps provided in the following paper:
[Seydoux et al., 2016](https://doi.org/10.1093/gji/ggv531). The steps are as follows:
1. Remove the mean of the data
2. Do a fourier transform of the data
3. Divide spectrum representation by rolling average
4. Do an inverse fourier transform
5. Divide the data the rolling average

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as ss

sys.path.append(os.path.join(os.path.dirname(""), os.pardir))
import coherence_analysis.utils as f

In [ ]:
file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160312000018.h5"
data, _ = f.load_brady_hdf5(file, normalize="no")

file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160312000048.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

data_noise = np.append(data, data2, axis=1)

# file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160312000118.h5"
# data2,_= f.load_brady_hdf5(file,normalize='no')

# data_noise = np.append(data_noise, data2[:, :20000], axis=1)

In [ ]:
sampling_rate = 1000
sample_interval = 1 / sampling_rate
num_samples = len(data_noise[0])

fsize = 15
tick_size = 12
nsensors = 200
start_ch = 1000
nchannels = 3000

data_noise = data_noise[
    start_ch : nchannels + start_ch : int(nchannels / nsensors)
]

### Compute FFT of the data

In [ ]:
spectra = np.fft.rfft(data_noise[:])

frequencies = np.fft.rfftfreq(num_samples, sample_interval)

In [ ]:
plt.plot(frequencies, np.abs(spectra[0]))
plt.title("Spectra of the signal for single sensor", fontsize=fsize)

#### Compute running average of the FFT data

In [ ]:
N = int(0.33 / (frequencies[1] - frequencies[0]))

running_avg = ss.fftconvolve(
    np.abs(spectra), np.ones((len(spectra), N)) / N, mode="same", axes=1
)

# 1d
# running_avg = np.convolve(np.abs(spectra[0]), np.ones(N)/N, mode='same')

plt.plot(running_avg[0].real)
plt.title("Running average of the signal for single sensor", fontsize=fsize)

### Divide the FFT data by rolling average

In [ ]:
spectral_whitened = spectra / running_avg.real

plt.plot(frequencies, spectral_whitened[0])
plt.title("Whitened signal for single sensor", fontsize=fsize)

#### Spectral whitened data in time

In [ ]:
whitened_time = np.fft.irfft(spectral_whitened)

plt.plot(data_noise[0], label="Original signal")
plt.plot(whitened_time[0], label="Whitened signal")
plt.title("Whitened signal for single sensor in time domain", fontsize=fsize)
plt.legend()

#### Compute running average for time domain normalization.

In [ ]:
N = int(1.25 * sampling_rate)

running_avg = ss.fftconvolve(
    np.abs(whitened_time),
    np.ones((len(whitened_time), N)) / N,
    mode="same",
    axes=1,
)

# 1d
# running_avg = np.convolve(np.abs(np.fft.irfft(spectral_whitened)), np.ones(N)/N, mode='same')

#### Divide the data by the running average for final preprocessed data

In [ ]:
preprocessed = whitened_time / running_avg

plt.plot(preprocessed[0])
plt.title(
    "Preprocessed signal for single sensor in time domain", fontsize=fsize
)

In [ ]:
pp = f.covariance_preprocessing(data_noise[2], sample_interval=sample_interval)

len(pp[0])

## Use preprocessed data for coherence analysis

In [ ]:
subwindow_len = 1
overlap = 0.5

In [ ]:
covariance, frequencies = f.covariance(
    preprocessed,
    subwindow_len,
    overlap,
    sample_interval=0.001,
)

In [ ]:
num_frames = covariance.shape[0]
eig_ratios_covariance = np.empty(num_frames)

for d in range(num_frames):
    eigenvals, _ = np.linalg.eig(covariance[d])
    eigenvals = np.sort(eigenvals)[::-1]
    eig_ratios_covariance[d] = eigenvals[0] / np.sum(eigenvals)

In [ ]:
plt.plot(frequencies[1:-250], eig_ratios_covariance[1:-250])
plt.title("Eigenvalue ratios of the covariance matrix", fontsize=fsize)

Compare covariance analysis with coherence analysis.

In [ ]:
coherence, frequencies = f.welch_coherence(
    data[start_ch : nchannels + start_ch : int(nchannels / nsensors)],
    subwindow_len,
    overlap,
    sample_interval=0.001,
)
eig_ratios_coherence = np.empty(num_frames)
for d in range(num_frames):
    eigenvals, _ = np.linalg.eig(coherence[d])
    eigenvals = np.sort(eigenvals)[::-1]
    eig_ratios_coherence[d] = eigenvals[0] / np.sum(eigenvals)

In [ ]:
plt.plot(frequencies[1:-250], eig_ratios_coherence[1:-250])
plt.title("Eigenvalue ratios for coherence matrices", fontsize=fsize)

In [ ]:
plt.plot(frequencies[1:-250], eig_ratios_coherence[1:-250], label="Coherence")
plt.plot(
    frequencies[1:-250], eig_ratios_covariance[1:-250], label="Covariance"
)
plt.legend()